# TRUSTDESC · bounded reproduction (recovered public source)

**Apple Silicon, local, no paid APIs.** This notebook walks through the project that is
checked out in this repository. It is a companion to the reports, not a replacement for them.

## What this notebook covers

| Section | What it does | Cost |
|---|---|---|
| 1. Environment | confirms the interpreter and repo layout, no writes outside the workspace | seconds |
| 2. Historical automatic pipeline | the earlier synthetic-fixture experiment (kept for the record) | opt-in, minutes |
| 3. Recovered corpus | rebuilding discovery / slicing state (skipped when data is present) | minutes, resumable |
| 4. DescGen | the local 4-bit model run, resumable | ~1 hour, **already done** |
| 5. DynVer | sandboxed execution of real tool bodies, resumable | ~15 min, **already done** |

## Read this first

* Every experiment reported in `reports/` has **already been executed**. Re-running the
  notebook is not required to reproduce the numbers; regenerating the figures and the
  analysis is (`scripts/generate_figures.py`, `scripts/analyze_statistics.py`).
* The **Run All** command is not recommended: cells 6 and 12 invoke long runs. Run cells
  individually and read each cell's comment before executing it.
* The notebook no longer restores an embedded copy of the source tree. That older behaviour
  would have overwritten newer evidence files; the checked-out repository is authoritative.
* What is and is not claimed: see `reports/KNOWN_LIMITATIONS.md` and
  `reports/PROJECT_SUMMARY.md`. The paper's 208-task benchmark was never recovered, so no
  task success rate appears anywhere in this notebook.


## 1 · Environment and repository layout


In [1]:
# 1 · Environment check (non-destructive).
# Verifies the interpreter and the repository layout this notebook depends on.
from pathlib import Path
import json, os, platform, sys

PROJECT = Path.cwd().resolve()
BASE = PROJECT / "trustdesc_local"          # scratch workspace, never the repo itself
BASE.mkdir(exist_ok=True)

REQUIRED = [
    "src/trustdesc/dynver/recovered.py",
    "src/trustdesc/sandbox/seatbelt.py",
    "scripts/run_dynver_recovered.py",
    "scripts/generate_figures.py",
    "scripts/analyze_statistics.py",
    "data/manifests/recovered_benchmark.json",
    "reports/dynver_recovered/results.json",
]
missing = [name for name in REQUIRED if not (PROJECT / name).exists()]
assert not missing, f"Run this notebook from the repository root; missing: {missing}"
assert sys.version_info >= (3, 11), sys.version
print("python", platform.python_version(), "|", platform.system(), platform.machine())
print("repository", PROJECT)
print("all", len(REQUIRED), "required paths present")


python 3.11.14 | Darwin arm64
repository /Users/macbramesha/.config/freebuff-desktop/local-projects/67df7954-841e-4034-be87-5ea8c73d55df/_repo
all 7 required paths present


## 2 · The earlier automatic pipeline (kept for the record, not required)

Before the recovered corpus existed, this project ran a small synthetic-fixture pipeline
(`scripts/run_local_reproduction.py`) on pure, deterministic tools. Those runs are still in
`reports/apple_recovery/`, `reports/apple_silicon_delivery/` and `reports/local_experiments.json`.
They are *not* the experiments reported in this checkpoint and re-running them is unnecessary.

The notebook no longer restores an embedded copy of the source tree: that behaviour would
have overwritten newer evidence files with an older snapshot.


In [2]:
# Confirm the historical artefacts are still present; no execution.
from pathlib import Path
import json
for rel in ["reports/local_experiments.json", "reports/COMPARISON.md"]:
    print(f"{rel:45s} {'present' if (PROJECT/rel).exists() else 'absent'}")
previous = sorted(p.name for p in (PROJECT/"reports").glob("apple_recovery")) 
print("apple_recovery/ contents:", previous[:3])


reports/local_experiments.json                present
reports/COMPARISON.md                         present
apple_recovery/ contents: ['apple_recovery']


## 3 · Recovered public-source experiment


In [3]:
print('SKIPPED in validation: recovered DescGen/DynVer driver already executed')

SKIPPED in validation: recovered DescGen/DynVer driver already executed


## 4 · Sandboxed behavioural verification (DynVer)


In [4]:
# Sandboxed behavioural verification (DynVer) on recovered tools.
# Requires macOS sandbox-exec and a passing isolation probe; resumable per task.
from pathlib import Path
import subprocess, sys, json
ROOT = Path.cwd()
import os
RUN_AGAIN = os.environ.get('TRUSTDESC_RUN_DYNVER') == '1'
existing = json.loads((ROOT/'reports/dynver_recovered/results.json').read_text())
if RUN_AGAIN:
    r = subprocess.run([sys.executable, 'scripts/run_dynver_recovered.py'], cwd=ROOT,
                       env={**os.environ, 'PYTHONPATH': str(ROOT/'src')})
    print('dynver exit', r.returncode)
else:
    print(f"Recorded run reused ({len(existing['units'])} units). "
          "Set TRUSTDESC_RUN_DYNVER=1 to re-execute; completed units are skipped.")
results = json.loads((ROOT/'reports/dynver_recovered/results.json').read_text())
print(results['conditions'])
print('isolated backend:', results['execution_backend'])


Recorded run reused (18 units). Set TRUSTDESC_RUN_DYNVER=1 to re-execute; completed units are skipped.
{'original': {'supported': 4, 'refuted': 2, 'inconclusive': 0}, 'full': {'supported': 4, 'refuted': 2, 'inconclusive': 0}, 'lite': {'supported': 4, 'refuted': 2, 'inconclusive': 0}}
isolated backend: macOS Seatbelt (deny default, scratch-only writes, network denied)


## 5 · Regenerate the figures and the statistics


In [5]:
# Cheap and deterministic: both scripts read only recorded evidence.
import subprocess, sys
from pathlib import Path
for script in ['scripts/analyze_statistics.py', 'scripts/generate_figures.py']:
    r = subprocess.run([sys.executable, script], cwd=Path.cwd())
    print(script, 'exit', r.returncode)
from IPython.display import Image, display
for name in ['fig1_dataset_coverage', 'fig5_execution_coverage', 'fig8_research_summary']:
    display(Image(filename=f'reports/figures/{name}.png', width=760))


{
 "descgen_yield": 0.3571,
 "dynver": {
  "original": 0.6667,
  "full": 0.6667,
  "lite": 0.6667
 },
 "identical_tasks": false,
 "agreement": {
  "original_vs_full": {
   "cells": 6,
   "identical_verdicts": 4,
   "agreement": 0.6667
  },
  "original_vs_lite": {
   "cells": 6,
   "identical_verdicts": 4,
   "agreement": 0.6667
  },
  "full_vs_lite": {
   "cells": 6,
   "identical_verdicts": 4,
   "agreement": 0.6667
  }
 }
}
scripts/analyze_statistics.py exit 0


wrote fig1_dataset_coverage
wrote fig2_descgen_yield
wrote fig3_dynver_verdicts
wrote fig4_condition_comparison
wrote fig5_execution_coverage
wrote fig6_failure_taxonomy
wrote fig7_project_progress
wrote fig8_research_summary
dependency probe: [('excel-mcp-server', 'openpyxl', 'missing'), ('mysql_mcp_server', 'mysql.connector', 'missing'), ('yfinance-mcp', 'yfinance', 'missing'), ('imagesorcery-mcp', 'cv2', 'missing'), ('arxiv-mcp-server', 'dateutil', 'available')]
scripts/generate_figures.py exit 0


## Results, and what they do not show

Recorded outcome of this checkpoint (see `reports/STATISTICAL_RESULTS.json`):

* **Corpus** — 12/12 paper-named servers at pinned commits; 30/30 Python tool slots sliced;
  22 TypeScript/JavaScript slots unsupported.
* **Description generation** — 10 valid descriptions from 28 attempts (35.7%, Wilson 95%
  20.7–54.2%). Rejections: 8 generation deadlines, 7 identifier normalisations, 2 token-cap
  overruns, 1 schema failure.
* **Sandboxed verification** — 18 executions across 3 tools and 3 description conditions;
  12 claims supported, 6 refuted, 0 inconclusive.
* **Comparison** — original developer text, TRUSTDESC-full and TRUSTDESC-lite each produced
  4 supported of 6. The conditions did not separate, and because each condition generated
  its own tasks, no paired test is reported.

**Not shown, because not measured:** any task success rate, any cost or latency comparison
with the paper, any attack or poisoning result, any TypeScript tool result. The paper's
208-task benchmark, its labels and its traces were never released, so this notebook cannot
reproduce them.

**Limits to keep in view:** 6 units per condition from 3 tools; one server has an adapter
shim layer; identical counts are not evidence of equivalence. Full discussion in
`reports/STATISTICAL_ANALYSIS.md`, `reports/EXPERIMENTAL_FINDINGS.md` and
`reports/KNOWN_LIMITATIONS.md`.
